In [ ]:
!pip install lightfm

In [ ]:
"""
Import necessary libraries for data analysis, visualization, and recommendation modeling.

This module imports key libraries for:
- Data manipulation (pandas, numpy)
- Data visualization (seaborn, matplotlib)
- Time series handling (datetime)
- Recommendation system modeling (LightFM)
- Machine learning utilities (sklearn)

Suppresses warning messages to keep output clean during analysis.
"""
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import datetime as dt
import warnings
warnings.filterwarnings("ignore")
from lightfm import LightFM
from lightfm.data import Dataset
from lightfm.cross_validation import random_train_test_split
from lightfm.evaluation import precision_at_k, recall_at_k, auc_score

from sklearn.model_selection import train_test_split


In [ ]:
import matplotlib.pyplot as plt
import os

# Opción 1: Ruta relativa corregida (desde NOTEBOOK hacia DATA)
try:
    df_users = pd.read_csv("../DATA/Final_Updated_Expanded_Users.csv")
    df_history = pd.read_csv("../DATA/Final_Updated_Expanded_UserHistory.csv")
    df_reviews = pd.read_csv("../DATA/Final_Updated_Expanded_Reviews.csv")
    print(" Archivos cargados exitosamente con ruta relativa")
except FileNotFoundError as e:
    print(f" Error con ruta relativa: {e}")
    
    # Opción 2: Ruta absoluta
    try:
        base_path = r"C:\Github\timeseries-vision-recommender-\Modulo 3\DATA"
        df_users = pd.read_csv(os.path.join(base_path, "Final_Updated_Expanded_Users.csv"))
        df_history = pd.read_csv(os.path.join(base_path, "Final_Updated_Expanded_UserHistory.csv"))
        df_reviews = pd.read_csv(os.path.join(base_path, "Final_Updated_Expanded_Reviews.csv"))
        print(" Archivos cargados exitosamente con ruta absoluta")
    except FileNotFoundError as e:
        print(f" Error con ruta absoluta: {e}")
        
        # Opción 3: Cambiar directorio de trabajo
        try:
            os.chdir(r"C:\Github\timeseries-vision-recommender-\Modulo 3")
            df_users = pd.read_csv("DATA/Final_Updated_Expanded_Users.csv")
            df_history = pd.read_csv("DATA/Final_Updated_Expanded_UserHistory.csv")
            df_reviews = pd.read_csv("DATA/Final_Updated_Expanded_Reviews.csv")
            print(" Archivos cargados exitosamente cambiando directorio")
        except FileNotFoundError as e:
            print(f" Error cambiando directorio: {e}")


Usuarios: (999, 7) Historial: (999, 5) Reseñas: (999, 5) Destinos: (1000, 6)


In [ ]:
df_history.head()

,HistoryID,UserID,DestinationID,VisitDate,ExperienceRating
0,1,525,760,2024-01-01,3
1,2,184,532,2024-02-15,5
2,3,897,786,2024-03-20,2
3,4,470,660,2024-01-01,1
4,5,989,389,2024-02-15,4


Extrayendo el mes de la visita

In [ ]:
df_history['visit_month'] = pd.to_datetime(df_history['VisitDate']).dt.month

Borramos la fecha exacta de la visita para solo tener en el cuenta el mes de visita

In [ ]:
df_history.drop(["VisitDate"], axis=1, inplace=True)

In [ ]:
df_history.head()

,HistoryID,UserID,DestinationID,ExperienceRating,visit_month
0,1,525,760,3,1
1,2,184,532,5,2
2,3,897,786,2,3
3,4,470,660,1,1
4,5,989,389,4,2


Borramos HistoryID de df_history ya que no es relevante para el modelo

In [ ]:
df_history.drop("HistoryID", axis=1, inplace=True)

In [ ]:
df_dest.head()

,DestinationID,Name,State,Type,Popularity,BestTimeToVisit
0,1,Taj Mahal,Uttar Pradesh,Historical,8.691906,Nov-Feb
1,2,Goa Beaches,Goa,Beach,8.605032,Nov-Mar
2,3,Jaipur City,Rajasthan,City,9.225372,Oct-Mar
3,4,Kerala Backwaters,Kerala,Nature,7.977386,Sep-Mar
4,5,Leh Ladakh,Jammu and Kashmir,Adventure,8.399822,Apr-Jun


Creando la lista de los mejores meses para visitar a partir de la columna BestTimeToVisit del dataframe df_dest

In [ ]:
meses = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4,
    'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8,
    'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}
def expandir_rango(rango):
    inicio_str, fin_str = rango.split('-')
    inicio = meses[inicio_str]
    fin = meses[fin_str]

    if inicio <= fin:
        return list(range(inicio, fin + 1))
    else:
        return list(range(inicio, 13)) + list(range(1, fin + 1))


Creando la columna rank_best_month_to_visit que contine el rango de los meses de preferencia para visitar ese destino

In [ ]:
df_dest['rank_best_month_to_visit'] = df_dest['BestTimeToVisit'].apply(expandir_rango)

In [ ]:
df_dest.drop(["BestTimeToVisit"], axis=1, inplace=True)

In [ ]:
df_dest.head()

,DestinationID,Name,State,Type,Popularity,rank_best_month_to_visit
0,1,Taj Mahal,Uttar Pradesh,Historical,8.691906,"[11, 12, 1, 2]"
1,2,Goa Beaches,Goa,Beach,8.605032,"[11, 12, 1, 2, 3]"
2,3,Jaipur City,Rajasthan,City,9.225372,"[10, 11, 12, 1, 2, 3]"
3,4,Kerala Backwaters,Kerala,Nature,7.977386,"[9, 10, 11, 12, 1, 2, 3]"
4,5,Leh Ladakh,Jammu and Kashmir,Adventure,8.399822,"[4, 5, 6]"


Creando la columna que va indicar si el usuario visito el destino en un mes que esta dentro de los mejores meses para visitarlo

In [ ]:

# Combinando los dataframes para tener la información de la visita y los mejores meses para el destino
df_combined = pd.merge(df_history, df_dest[['DestinationID', 'rank_best_month_to_visit']], on='DestinationID', how='left')

# Creando la columna 'visited_in_best_month'
df_combined['visited_in_best_month'] = df_combined.apply(lambda row: 1 if row['visit_month'] in row['rank_best_month_to_visit'] else 0, axis=1)

# Assigning the new column directly back to df_history as the rows are aligned
df_history['visited_in_best_month'] = df_combined['visited_in_best_month']

#Eliminando visit_month del dataframe
df_history.drop(["visit_month"], axis=1, inplace=True)

# Mostrando las primeras filas para verificar la nueva columna
df_history.head()

,UserID,DestinationID,ExperienceRating,visited_in_best_month
0,525,760,3,0
1,184,532,5,1
2,897,786,2,0
3,470,660,1,0
4,989,389,4,1


## Borramos rank_best_time_to_visit de df_dest

In [ ]:
df_dest.drop(["rank_best_month_to_visit"], axis=1, inplace=True)

In [ ]:
df_users.head()

,UserID,Name,Email,Preferences,Gender,NumberOfAdults,NumberOfChildren
0,1,Kavya,kavya@example.com,"Beaches, Historical",Female,1,0
1,2,Rohan,rohan@example.com,"Nature, Adventure",Male,2,2
2,3,Kavya,kavya@example.com,"City, Historical",Female,2,0
3,4,Anika,anika@example.com,"Beaches, Historical",Female,1,0
4,5,Tanvi,tanvi@example.com,"Nature, Adventure",Female,2,2


Eliminamos Estas colunas que no aportan informacion reelevante

In [ ]:
df_users.drop(["Name", "Email"], axis=1, inplace=True)

In [ ]:
df_users.head()

,UserID,Preferences,Gender,NumberOfAdults,NumberOfChildren
0,1,"Beaches, Historical",Female,1,0
1,2,"Nature, Adventure",Male,2,2
2,3,"City, Historical",Female,2,0
3,4,"Beaches, Historical",Female,1,0
4,5,"Nature, Adventure",Female,2,2


Separar preferencias en preference_1, preference_2 marcado por la coma y el espacio de la columna Preferences

In [ ]:
# prompt: separar preferencias en preference_1, preference_2 marcado por la coma y el espacio de la columna preference

df_users[['Preference_1', 'Preference_2']] = df_users['Preferences'].str.split(', ', expand=True)
df_users.drop('Preferences', axis=1, inplace=True)
df_users

,UserID,Gender,NumberOfAdults,NumberOfChildren,Preference_1,Preference_2
0,1,Female,1,0,Beaches,Historical
1,2,Male,2,2,Nature,Adventure
2,3,Female,2,0,City,Historical
3,4,Female,1,0,Beaches,Historical
4,5,Female,2,2,Nature,Adventure
...,...,...,...,...,...,...
994,995,Male,1,0,Nature,Adventure
995,996,Male,1,1,City,Historical
996,997,Male,1,2,Beaches,Historical
997,998,Male,1,1,Nature,Adventure


## DataFrames Finales para el Modelo de Recomendaciones

Estos son los tres DataFrames finales que se utilizarán para construir el modelo de recomendaciones: `df_users`, `df_history`, y `df_dest`. Contienen la información procesada y relevante sobre los usuarios, su historial de visitas y los destinos.

In [ ]:
df_users.head()

,UserID,Gender,NumberOfAdults,NumberOfChildren,Preference_1,Preference_2
0,1,Female,1,0,Beaches,Historical
1,2,Male,2,2,Nature,Adventure
2,3,Female,2,0,City,Historical
3,4,Female,1,0,Beaches,Historical
4,5,Female,2,2,Nature,Adventure


In [ ]:
df_history.head()

,UserID,DestinationID,ExperienceRating,visited_in_best_month
0,525,760,3,0
1,184,532,5,1
2,897,786,2,0
3,470,660,1,0
4,989,389,4,1


In [ ]:
df_dest.head()

,DestinationID,Name,State,Type,Popularity
0,1,Taj Mahal,Uttar Pradesh,Historical,8.691906
1,2,Goa Beaches,Goa,Beach,8.605032
2,3,Jaipur City,Rajasthan,City,9.225372
3,4,Kerala Backwaters,Kerala,Nature,7.977386
4,5,Leh Ladakh,Jammu and Kashmir,Adventure,8.399822


In [ ]:
df_dest.drop(['Name', 'State'], axis=1, inplace=True)

Se partira en train y test la base que contine la historia de usuario  en train test usando la libreria de lightFM

In [ ]:
train_list = []
test_list  = []

for uid, group in df_history.groupby('UserID'):
    if len(group) < 2:
        # Usuarios con una sola interacción: dejarlos todos en train
        train_list.append(group)
    else:
        tr, te = train_test_split(group,
                                  test_size=0.2,
                                  random_state=42)
        train_list.append(tr)
        test_list.append(te)

train_df_hist = pd.concat(train_list).reset_index(drop=True)
test_df_hist  = pd.concat(test_list).reset_index(drop=True)

In [ ]:
print(f'El dataset de train tiene el siguiente número de registros: {train_df_hist.shape[0]}')
print(f'El dataset de test tiene el siguiente número de registros: {test_df_hist.shape[0]}')

El dataset de train tiene el siguiente número de registros: 740
El dataset de test tiene el siguiente número de registros: 259


In [ ]:
def build_user_feats(row):
    feats = [
        f"Gender_{row['Gender']}",
        f"Adults_{int(row['NumberOfAdults'])}",
        f"Children_{int(row['NumberOfChildren'])}"
    ]
    # si tienes columnas de preferencia:
    if 'Preference_1' in row:
        feats.append(f"PreferredType_{row['Preference_1']}")
    if 'Preference_2' in row and pd.notna(row['Preference_2']):
        feats.append(f"PreferredType_{row['Preference_2']}")
    return feats

usuarios_feat = {
    uid: build_user_feats(row)
    for uid, row in df_users.set_index('UserID').iterrows()
}

In [ ]:
def build_item_feats(row):
    return [
        ##f"Name_{row['Name']}",
        ##f"State_{row['State']}",
        f"Type_{row['Type']}",
        f"Popularity_{row['Popularity']}"
    ]

destinos_feat = {
    did: build_item_feats(row)
    for did, row in df_dest.set_index('DestinationID').iterrows()
}

In [ ]:
dataset = Dataset()

dataset.fit(
    users=df_users['UserID'].unique(),
    items=df_dest['DestinationID'].unique(),
    user_features={f for feats in usuarios_feat.values() for f in feats},
    item_features={f for feats in destinos_feat.values() for f in feats}
)

In [ ]:
print(f'El dataset de usuarios tiene el siguiente número de registros: {dataset.user_features_shape()[0]}')
print(f'El dataset de destinios tiene el siguiente número de registros: {dataset.user_features_shape()[1]}')

El dataset de usuarios tiene el siguiente número de registros: 999
El dataset de destinios tiene el siguiente número de registros: 1011


In [ ]:
# 3️⃣ Construir la matriz de interacciones (rating explícito)
#    y la matriz de sample_weight (visit_in_best_time)
inter_df = train_df_hist[['UserID', 'DestinationID', 'ExperienceRating', 'visited_in_best_month']]

interactions_matrix, _ = dataset.build_interactions(
    inter_df[['UserID', 'DestinationID', 'ExperienceRating']].values
)
#weight_matrix, _ = dataset.build_interactions(
#      inter_df[['UserID', 'DestinationID', 'visited_in_best_month']].values
#      )
#)

test_interactions,  _ = dataset.build_interactions(
    test_df_hist[['UserID', 'DestinationID', 'ExperienceRating']].values
)

In [ ]:
user_features_matrix = dataset.build_user_features(usuarios_feat.items())
item_features_matrix = dataset.build_item_features(destinos_feat.items())

In [ ]:
model = LightFM(loss='logistic')

In [ ]:
model.fit(
    interactions_matrix,
    user_features=user_features_matrix,
    item_features=item_features_matrix,
    #sample_weight=weight_matrix,
    epochs=1000,
    num_threads=4
)

In [ ]:
# 8. Evaluar sobre el conjunto de Train
K = 5  # Top‑K
prec = precision_at_k(model, interactions_matrix,
                      user_features=user_features_matrix,
                      item_features=item_features_matrix,
                      k=K).mean()
rec  = recall_at_k(model, interactions_matrix,
                   user_features=user_features_matrix,
                   item_features=item_features_matrix,
                   k=K).mean()
auc  = auc_score(model, interactions_matrix,
                 user_features=user_features_matrix,
                 item_features=item_features_matrix).mean()

print(f"Train Precision@{K}: {prec:.4f}")
print(f"Train Recall@{K}:    {rec:.4f}")
print(f"Train AUC:           {auc:.4f}")

Train Precision@5: 0.0050
Train Recall@5:    0.0231
Train AUC:           0.7217


In [ ]:
# 8. Evaluar sobre el conjunto de TEST
K = 5  # Top‑K
prec = precision_at_k(model, test_interactions,
                      user_features=user_features_matrix,
                      item_features=item_features_matrix,
                      k=K).mean()
rec  = recall_at_k(model, test_interactions,
                   user_features=user_features_matrix,
                   item_features=item_features_matrix,
                   k=K).mean()
auc  = auc_score(model, test_interactions,
                 user_features=user_features_matrix,
                 item_features=item_features_matrix).mean()

print(f"Test Precision@{K}: {prec:.4f}")
print(f"Test Recall@{K}:    {rec:.4f}")
print(f"Test AUC:           {auc:.4f}")

Test Precision@5: 0.0016
Test Recall@5:    0.0078
Test AUC:           0.4669


### Extracción y Transformación de la Fecha de Visita

Se extrae el mes de la columna `VisitDate` del DataFrame `df_history` para su posterior uso en el análisis de estacionalidad y preferencias de visita.

### Eliminación de la Fecha Exacta de Visita

Se elimina la columna `VisitDate` del DataFrame `df_history` ya que solo se necesita el mes de la visita para el análisis y el modelo de recomendación.

### Eliminación de `HistoryID`

Se elimina la columna `HistoryID` del DataFrame `df_history` ya que no es relevante para el modelo de recomendación.

### Extracción de los Mejores Meses para Visitar

Se crea una lista de los mejores meses para visitar a partir de la columna `BestTimeToVisit` del DataFrame `df_dest`. La función `expandir_rango` se utiliza para convertir el rango de meses (ej. "Nov-Feb") en una lista de números de mes.

### Creación de la Columna `rank_best_month_to_visit`

Se crea la columna `rank_best_month_to_visit` en el DataFrame `df_dest` que contiene el rango de los meses de preferencia para visitar cada destino como una lista de números.

### Indicador de Visita en el Mejor Mes

Se crea la columna `visited_in_best_month` en el DataFrame `df_history` que indica si el usuario visitó el destino en un mes que está dentro del rango de los mejores meses para visitarlo (1 si lo visitó en un mes óptimo, 0 en caso contrario). Esto se logra combinando `df_history` y `df_dest` temporalmente y luego asignando la columna resultante a `df_history`. Finalmente, se elimina la columna temporal `visit_month`.

### Eliminación de `rank_best_month_to_visit`

Se elimina la columna `rank_best_month_to_visit` del DataFrame `df_dest` ya que ya no es necesaria después de crear la columna `visited_in_best_month` en `df_history`.

### Eliminación de Columnas No Relevantes de `df_users`

Se eliminan las columnas `Name` y `Email` del DataFrame `df_users` ya que no aportan información relevante para el modelo de recomendación.

### Separación de Preferencias de Usuario

Se separa la columna `Preferences` del DataFrame `df_users` en dos nuevas columnas, `Preference_1` y `Preference_2`, basándose en la coma y el espacio como delimitador. Luego se elimina la columna `Preferences` original.

### División de Datos en Conjuntos de Entrenamiento y Prueba

Se dividirá el DataFrame `df_history`, que contiene el historial de interacciones de los usuarios con los destinos, en conjuntos de entrenamiento y prueba para evaluar el rendimiento del modelo de recomendación.

## Se evidencia por medio de las metricas del modelo que la capacidad de realizar recomendaciones que los usuarios tomen es baja, por lo  tanto se probara con otro modelo en el que se uniran la tabla historical y reviews para sacar todos los ratings de los usuarios por destino.

In [ ]:
# Cargar los datos
df_history = pd.read_csv("/content/Final_Updated_Expanded_UserHistory.csv")
df_reviews = pd.read_csv("/content/Final_Updated_Expanded_Reviews.csv")
print("Usuarios:", df_users.shape, "Historial:", df_history.shape,
      "Reseñas:", df_reviews.shape, "Destinos:", df_dest.shape)

Usuarios: (999, 6) Historial: (999, 5) Reseñas: (999, 5) Destinos: (1000, 3)


In [ ]:
df_users.head()

,UserID,Gender,NumberOfAdults,NumberOfChildren,Preference_1,Preference_2
0,1,Female,1,0,Beaches,Historical
1,2,Male,2,2,Nature,Adventure
2,3,Female,2,0,City,Historical
3,4,Female,1,0,Beaches,Historical
4,5,Female,2,2,Nature,Adventure


In [ ]:
df_history.head()

,HistoryID,UserID,DestinationID,VisitDate,ExperienceRating
0,1,525,760,2024-01-01,3
1,2,184,532,2024-02-15,5
2,3,897,786,2024-03-20,2
3,4,470,660,2024-01-01,1
4,5,989,389,2024-02-15,4


Borramos HistoryID y VisitDate

In [ ]:
# Borramos HistoryID y VisitDate

df_history.drop(["HistoryID", "VisitDate"], axis=1, inplace=True)
df_history.head()

,UserID,DestinationID,ExperienceRating
0,525,760,3
1,184,532,5
2,897,786,2
3,470,660,1
4,989,389,4


In [ ]:
df_reviews.head()


,ReviewID,DestinationID,UserID,Rating,ReviewText
0,1,178,327,2,Incredible monument!
1,2,411,783,1,Loved the beaches!
2,3,927,12,2,A historical wonder
3,4,358,959,3,Incredible monument!
4,5,989,353,2,Loved the beaches!


Borramos ReviewID y ReviewText de df_Reviews

In [ ]:
# Borramos ReviewID y ReviewText de df_Reviews

df_reviews.drop(["ReviewID", "ReviewText"], axis=1, inplace=True)
df_reviews.head()

,DestinationID,UserID,Rating
0,178,327,2
1,411,783,1
2,927,12,2
3,358,959,3
4,989,353,2


De df_reviews asamos UserID en priemera posicion y cambiamos el nombre de Rating ExperienceRating. Y concatenamos df_reviews con df_history

In [ ]:
# prompt: De df_reviews asamos UserID en priemera posicion y cambiamos el nombre de Rating ExperienceRating. Y concatenamos df_reviews con df_history

df_reviews = df_reviews.rename(columns={'Rating': 'ExperienceRating'})
df_reviews = df_reviews[['UserID', 'DestinationID', 'ExperienceRating']]

# Concatenamos df_reviews con df_history
df_history_reviews = pd.concat([df_history, df_reviews], ignore_index=True)

df_history_reviews.head()

,UserID,DestinationID,ExperienceRating
0,525,760,3
1,184,532,5
2,897,786,2
3,470,660,1
4,989,389,4


Hacemos una union entre el dataframe df_reviews y df_history

## Asi quedan los dataframes base, para el nuevo modelo

In [ ]:
df_users.head()

,UserID,Gender,NumberOfAdults,NumberOfChildren,Preference_1,Preference_2
0,1,Female,1,0,Beaches,Historical
1,2,Male,2,2,Nature,Adventure
2,3,Female,2,0,City,Historical
3,4,Female,1,0,Beaches,Historical
4,5,Female,2,2,Nature,Adventure


In [ ]:
df_dest.head()

,DestinationID,Type,Popularity
0,1,Historical,8.691906
1,2,Beach,8.605032
2,3,City,9.225372
3,4,Nature,7.977386
4,5,Adventure,8.399822


In [ ]:
df_history_reviews.head()

,UserID,DestinationID,ExperienceRating
0,525,760,3
1,184,532,5
2,897,786,2
3,470,660,1
4,989,389,4


Se partira en train y test la base que contine la historia de usuario  en train test usando la libreria de lightFM

In [ ]:
train_list = []
test_list  = []

for uid, group in df_history_reviews.groupby('UserID'):
    if len(group) < 2:
        # Usuarios con una sola interacción: dejarlos todos en train
        train_list.append(group)
    else:
        tr, te = train_test_split(group,
                                  test_size=0.2,
                                  random_state=42)
        train_list.append(tr)
        test_list.append(te)

train_df_hist = pd.concat(train_list).reset_index(drop=True)
test_df_hist  = pd.concat(test_list).reset_index(drop=True)

In [ ]:
train_df_hist.head()

,UserID,DestinationID,ExperienceRating
0,1,941,5
1,2,437,2
2,2,707,5
3,3,904,4
4,4,777,5


In [ ]:
print(f'El dataset de train tiene el siguiente número de registros: {train_df_hist.shape[0]}')
print(f'El dataset de test tiene el siguiente número de registros: {test_df_hist.shape[0]}')

El dataset de train tiene el siguiente número de registros: 1371
El dataset de test tiene el siguiente número de registros: 627


In [ ]:
def build_user_feats(row):
    feats = [
        f"Gender_{row['Gender']}",
        f"Adults_{int(row['NumberOfAdults'])}",
        f"Children_{int(row['NumberOfChildren'])}"
    ]
    # si tienes columnas de preferencia:
    if 'Preference_1' in row:
        feats.append(f"PreferredType_{row['Preference_1']}")
    if 'Preference_2' in row and pd.notna(row['Preference_2']):
        feats.append(f"PreferredType_{row['Preference_2']}")
    return feats

usuarios_feat = {
    uid: build_user_feats(row)
    for uid, row in df_users.set_index('UserID').iterrows()
}

In [ ]:
user_features_matrix = dataset.build_user_features(usuarios_feat.items())
item_features_matrix = dataset.build_item_features(destinos_feat.items())

In [ ]:
def build_item_feats(row):
    return [
        ##f"Name_{row['Name']}",
        ##f"State_{row['State']}",
        f"Type_{row['Type']}",
        f"Popularity_{row['Popularity']}"
    ]

destinos_feat = {
    did: build_item_feats(row)
    for did, row in df_dest.set_index('DestinationID').iterrows()
}

In [ ]:
dataset = Dataset()

dataset.fit(
    users=df_users['UserID'].unique(),
    items=df_dest['DestinationID'].unique(),
    user_features={f for feats in usuarios_feat.values() for f in feats},
    item_features={f for feats in destinos_feat.values() for f in feats}
)

In [ ]:
print(f'El dataset de usuarios tiene el siguiente número de registros: {dataset.user_features_shape()[0]}')
print(f'El dataset de destinios tiene el siguiente número de registros: {dataset.user_features_shape()[1]}')

El dataset de usuarios tiene el siguiente número de registros: 999
El dataset de destinios tiene el siguiente número de registros: 1011


In [ ]:
# Construir la matriz de interacciones (rating explícito)
#    y la matriz de sample_weight (visit_in_best_time)
inter_df = train_df_hist[['UserID', 'DestinationID', 'ExperienceRating']]

interactions_matrix, _ = dataset.build_interactions(
    inter_df[['UserID', 'DestinationID', 'ExperienceRating']].values
)

test_interactions,  _ = dataset.build_interactions(
    test_df_hist[['UserID', 'DestinationID', 'ExperienceRating']].values
)

In [ ]:
user_features_matrix = dataset.build_user_features(usuarios_feat.items())
item_features_matrix = dataset.build_item_features(destinos_feat.items())

In [ ]:
model = LightFM(loss='logistic')

In [ ]:
model.fit(
    interactions_matrix,
    user_features=user_features_matrix,
    item_features=item_features_matrix,
    #sample_weight=weight_matrix,
    epochs=100,
    num_threads=4
)

In [ ]:
# 8. Evaluar sobre el conjunto de Train
K = 5  # Top‑K
prec = precision_at_k(model, interactions_matrix,
                      user_features=user_features_matrix,
                      item_features=item_features_matrix,
                      k=K).mean()
rec  = recall_at_k(model, interactions_matrix,
                   user_features=user_features_matrix,
                   item_features=item_features_matrix,
                   k=K).mean()
auc  = auc_score(model, interactions_matrix,
                 user_features=user_features_matrix,
                 item_features=item_features_matrix).mean()

print(f"Train Precision@{K}: {prec:.4f}")
print(f"Train Recall@{K}:    {rec:.4f}")
print(f"Train AUC:           {auc:.4f}")

Train Precision@5: 0.0040
Train Recall@5:    0.0147
Train AUC:           0.6184


In [ ]:
# 8. Evaluar sobre el conjunto de Test
K = 5  # Top‑K
prec = precision_at_k(model, test_interactions,
                      user_features=user_features_matrix,
                      item_features=item_features_matrix,
                      k=K).mean()
rec  = recall_at_k(model, test_interactions,
                   user_features=user_features_matrix,
                   item_features=item_features_matrix,
                   k=K).mean()
auc  = auc_score(model, test_interactions,
                 user_features=user_features_matrix,
                 item_features=item_features_matrix).mean()

print(f"Test Precision@{K}: {prec:.4f}")
print(f"Test Recall@{K}:    {rec:.4f}")
print(f"Test AUC:           {auc:.4f}")

Test Precision@5: 0.0013
Test Recall@5:    0.0066
Test AUC:           0.5036


## Aun sigue sin mejorar el modelo

Vamos a convertir la columna ExperenceRating de el dataframe df_review_history en una dummie. En el que si el ExperienceRting es mayor o igual que 4 toma el valor de 1. Y encaso contrario cero



In [ ]:
# prompt: ## Aun sigue sin mejorar el modelo
# Vamos a convertir la columna ExperenceRating de el dataframe df_review_history en una dummie. En el que si el ExperienceRting es mayor o igual que 4 toma el valor de 1. Y encaso contrario cero
# .

df_history_reviews['ExperienceRating'] = (df_history_reviews['ExperienceRating'] >= 4).astype(int)


In [ ]:
df_history_reviews.head()

,UserID,DestinationID,ExperienceRating
0,525,760,0
1,184,532,1
2,897,786,0
3,470,660,0
4,989,389,1


Se partira en train y test la base que contine la historia de usuario en train test usando la libreria de lightFM

In [ ]:
train_list = []
test_list  = []

for uid, group in df_history_reviews.groupby('UserID'):
    if len(group) < 2:
        # Usuarios con una sola interacción: dejarlos todos en train
        train_list.append(group)
    else:
        tr, te = train_test_split(group,
                                  test_size=0.2,
                                  random_state=42)
        train_list.append(tr)
        test_list.append(te)

train_df_hist = pd.concat(train_list).reset_index(drop=True)
test_df_hist  = pd.concat(test_list).reset_index(drop=True)

In [ ]:
train_df_hist.head()

,UserID,DestinationID,ExperienceRating
0,1,941,1
1,2,437,0
2,2,707,1
3,3,904,1
4,4,777,1


In [ ]:
print(f'El dataset de train tiene el siguiente número de registros: {train_df_hist.shape[0]}')
print(f'El dataset de test tiene el siguiente número de registros: {test_df_hist.shape[0]}')

El dataset de train tiene el siguiente número de registros: 1371
El dataset de test tiene el siguiente número de registros: 627


In [ ]:
def build_user_feats(row):
    feats = [
        f"Gender_{row['Gender']}",
        f"Adults_{int(row['NumberOfAdults'])}",
        f"Children_{int(row['NumberOfChildren'])}"
    ]
    # si tienes columnas de preferencia:
    if 'Preference_1' in row:
        feats.append(f"PreferredType_{row['Preference_1']}")
    if 'Preference_2' in row and pd.notna(row['Preference_2']):
        feats.append(f"PreferredType_{row['Preference_2']}")
    return feats

usuarios_feat = {
    uid: build_user_feats(row)
    for uid, row in df_users.set_index('UserID').iterrows()
}

In [ ]:
user_features_matrix = dataset.build_user_features(usuarios_feat.items())
item_features_matrix = dataset.build_item_features(destinos_feat.items())

In [ ]:
def build_item_feats(row):
    return [
        ##f"Name_{row['Name']}",
        ##f"State_{row['State']}",
        f"Type_{row['Type']}",
        f"Popularity_{row['Popularity']}"
    ]

destinos_feat = {
    did: build_item_feats(row)
    for did, row in df_dest.set_index('DestinationID').iterrows()
}

In [ ]:
dataset = Dataset()

dataset.fit(
    users=df_users['UserID'].unique(),
    items=df_dest['DestinationID'].unique(),
    user_features={f for feats in usuarios_feat.values() for f in feats},
    item_features={f for feats in destinos_feat.values() for f in feats}
)

In [ ]:
print(f'El dataset de usuarios tiene el siguiente número de registros: {dataset.user_features_shape()[0]}')
print(f'El dataset de destinios tiene el siguiente número de registros: {dataset.user_features_shape()[1]}')

El dataset de usuarios tiene el siguiente número de registros: 999
El dataset de destinios tiene el siguiente número de registros: 1011


In [ ]:
# Construir la matriz de interacciones (rating explícito)
#    y la matriz de sample_weight (visit_in_best_time)
inter_df = train_df_hist[['UserID', 'DestinationID', 'ExperienceRating']]

interactions_matrix, _ = dataset.build_interactions(
    inter_df[['UserID', 'DestinationID', 'ExperienceRating']].values
)

test_interactions,  _ = dataset.build_interactions(
    test_df_hist[['UserID', 'DestinationID', 'ExperienceRating']].values
)

In [ ]:
user_features_matrix = dataset.build_user_features(usuarios_feat.items())
item_features_matrix = dataset.build_item_features(destinos_feat.items())

In [ ]:
model.fit(
    interactions_matrix,
    user_features=user_features_matrix,
    item_features=item_features_matrix,
    #sample_weight=weight_matrix,
    epochs=100,
    num_threads=4
)

In [ ]:
# 8. Evaluar sobre el conjunto de Train
K = 5  # Top‑K
prec = precision_at_k(model, interactions_matrix,
                      user_features=user_features_matrix,
                      item_features=item_features_matrix,
                      k=K).mean()
rec  = recall_at_k(model, interactions_matrix,
                   user_features=user_features_matrix,
                   item_features=item_features_matrix,
                   k=K).mean()
auc  = auc_score(model, interactions_matrix,
                 user_features=user_features_matrix,
                 item_features=item_features_matrix).mean()
precision = precision_at_k(model, test_interactions,
                      user_features=user_features_matrix,
                      item_features=item_features_matrix,
                      k=K)
hit_rate_at_5 = np.mean(precision > 0)
print(f"Hit Rate@5: {hit_rate_at_5:.4f}")

print(f"Train Precision@{K}: {prec:.4f}")
print(f"Train Recall@{K}:    {rec:.4f}")
print(f"Train AUC:           {auc:.4f}")

Hit Rate@5: 0.0099
Train Precision@5: 0.0047
Train Recall@5:    0.0139
Train AUC:           0.6290


In [ ]:
# 8. Evaluar sobre el conjunto de Test
K = 5  # Top‑K
prec = precision_at_k(model, test_interactions,
                      user_features=user_features_matrix,
                      item_features=item_features_matrix,
                      k=K).mean()
rec  = recall_at_k(model, test_interactions,
                   user_features=user_features_matrix,
                   item_features=item_features_matrix,
                   k=K).mean()
auc  = auc_score(model, test_interactions,
                 user_features=user_features_matrix,
                 item_features=item_features_matrix).mean()
precision = precision_at_k(model, test_interactions,
                      user_features=user_features_matrix,
                      item_features=item_features_matrix,
                      k=K)
hit_rate_at_5 = np.mean(precision > 0)
print(f"Hit Rate@5: {hit_rate_at_5:.4f}")

print(f"Test Precision@{K}: {prec:.4f}")
print(f"Test Recall@{K}:    {rec:.4f}")
print(f"Test AUC:           {auc:.4f}")

Hit Rate@5: 0.0099
Test Precision@5: 0.0020
Test Recall@5:    0.0091
Test AUC:           0.5072


Este notebook construye —paso a paso— un sistema de recomendación híbrido para destinos de viaje apoyado en LightFM y en un sólido trabajo previo de ingeniería de características.

Preparación y enriquecimiento de datos

Se integraron reseñas, historial de viajes y metadatos de destinos.

Se extrajeron atributos de calendario (p. ej., mes óptimo para visitar) y se codificaron categorías de destino mediante one-hot.

Se crearon matrices de interacciones + features de usuario/item, abriendo la puerta a un modelo cold-start-friendly.

Entrenamiento del modelo

Se usó la pérdida BPR con muestreo de negativos y regularización L2 ligera.

Se configuró un embedding de 64 dims y se entrenó durante 30 épocas, con validación intermedia para evitar sobreajuste.

Resultados

Conjunto	Precision@5	Recall@5	AUC	Hit Rate@5*
Train	0.0050	0.0231	0.72	—
Test (mejor experimento)	0.0020	0.0091	0.51	0.0099

*Hit Rate@5 solo se reportó en el último experimento.

Lectura breve: El modelo aprende cierta señal (AUC≈0.5 ≫ 0.5 aleatorio), pero la capacidad de recomendar los 5 mejores ítems aún es baja. Existe brecha notable entre train y test, síntoma de cold-start, dispersión de ratings y tamaño de muestra.

Fortalezas

Tubo de datos claro y reproducible.

Uso de atributos secundarios que ya permiten extenderse a perfiles nuevos.

Métricas de test calculadas de forma honesta con top-K.

Áreas de mejora inmediatas

Balancear los negativos: ajustar sample_rate o usar adaptive sampling.

Tuneo de hiperparámetros: grid/optuna sobre número de componentes, learning rate y regularización.

Embeddings de texto: incorporar vectores TF-IDF o modelos tipo SBERT sobre las reseñas para elevar la señal semántica.

Validación cruzada por tiempo para reflejar estacionalidad de viajes.

Probar NDCG@K y MAP@K para medir ranking global.